# Notebook 02 - Limpeza e preparação dos dados

**Objetivo:** transformar a amostra filtrada no notebook 01 em um dataset pronto para a triagem de importância (notebook 03) e para a modelagem (notebook 05).

**Posição no fluxo:**

> 01 Inspeção → **02 Limpeza** → 03 Importância → 04 Justificativa + EDA focado → 05 Modelo

A limpeza vem **logo depois da inspeção** porque só com os dados tratados a comparação de R² entre blocos candidatos (notebook 03) tem significado: outliers não-tratados de salário inflam variância e mascaram qual variável X realmente carrega informação sobre o salário.

**Entrada:** `data/processed/df_filtrado_inspecao.csv` (18.191 respondentes)
**Saída:** `data/processed/df_limpo.csv`

**Tratamentos aplicados:**
1. `YearsCode`: imputação de nulos pela mediana e clip em 50 anos.
2. **`Country`: redução para top 5 + "Outros"** — feita ANTES dos outliers porque o passo seguinte é estratificado por país.
3. **Outliers de salário: removidos por IQR DENTRO de cada país do Top 5** — um salário "normal" nos EUA seria outlier na Índia, e vice-versa. Em "Outros", aplica-se um filtro global suave (USD 1k–500k) para descartar lixo extremo.
4. `DevType`: agrupamento em macro-categorias (backend/frontend/fullstack/mobile/data_ml) e exclusão de cargos não-desenvolvedores.
5. `EdLevel`: padronização e top 5 + "Outros".
6. Dummies das **5 linguagens mais frequentes**.
7. Criação de `nivel_experiencia` (Junior/Pleno/Sênior).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configurações visuais consistentes com notebook 01
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

# Caminhos
PASTA_DADOS = Path("../data/processed")
PASTA_FIGURAS = Path("../output/figures")
PASTA_TABELAS = Path("../output/tables")

## 1. Carregamento do dataset filtrado

Carrega o CSV produzido pelo notebook 01 e confirma dimensões e colunas.

In [2]:
df = pd.read_csv(PASTA_DADOS / "df_filtrado_inspecao.csv")

print(f"Linhas: {len(df):,}")
print(f"Colunas: {df.shape[1]}")
print(f"\nColunas disponíveis:")
print(df.columns.tolist())
df.head()

Linhas: 18,191
Colunas: 9

Colunas disponíveis:
['ResponseId', 'MainBranch', 'EdLevel', 'Employment', 'YearsCode', 'DevType', 'Country', 'LanguageHaveWorkedWith', 'ConvertedCompYearly']


,ResponseId,MainBranch,EdLevel,Employment,YearsCode,DevType,Country,LanguageHaveWorkedWith,ConvertedCompYearly
0,1,I am a developer by profession,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,14.0,"Developer, mobile",Ukraine,Bash/Shell (all shells);Dart;SQL,61256.0
1,2,I am a developer by profession,"Associate degree (A.A., A.S., etc.)",Employed,10.0,"Developer, back-end",Netherlands,Java,104413.0
2,3,I am a developer by profession,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Independent contractor, freelancer, or self-em...",12.0,"Developer, front-end",Ukraine,Dart;HTML/CSS;JavaScript;TypeScript,53061.0
3,4,I am a developer by profession,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,5.0,"Developer, back-end",Ukraine,Java;Kotlin;SQL,36197.0
4,5,I am a developer by profession,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Independent contractor, freelancer, or self-em...",22.0,Engineering manager,Ukraine,C;C#;C++;Delphi;HTML/CSS;Java;JavaScript;Lua;P...,60000.0


## 2. Diagnóstico consolidado

Antes de aplicar qualquer tratamento, confirma em uma única tela os problemas a serem resolvidos.

In [3]:
print("=" * 60)
print("DIAGNÓSTICO INICIAL")
print("=" * 60)

print(f"\n[1] YearsCode — tipo atual: {df['YearsCode'].dtype}")
print(f"    Valores nulos: {df['YearsCode'].isna().sum()}")

print(f"\n[2] ConvertedCompYearly — cauda longa?")
print(f"    Min: USD {df['ConvertedCompYearly'].min():,.0f}")
print(f"    Mediana: USD {df['ConvertedCompYearly'].median():,.0f}")
print(f"    Max: USD {df['ConvertedCompYearly'].max():,.0f}")

print(f"\n[3] Country — valores únicos: {df['Country'].nunique()}")
print(f"\n[4] DevType — valores únicos: {df['DevType'].nunique()}")
print(f"\n[5] EdLevel — valores únicos: {df['EdLevel'].nunique()}")

DIAGNÓSTICO INICIAL

[1] YearsCode — tipo atual: float64
    Valores nulos: 45

[2] ConvertedCompYearly — cauda longa?
    Min: USD 1
    Mediana: USD 80,254
    Max: USD 33,552,715

[3] Country — valores únicos: 153

[4] DevType — valores únicos: 32

[5] EdLevel — valores únicos: 8


## 3. Tratamento de YearsCode

A coluna foi carregada como `float64`. Verificamos se há valores especiais escondidos e tratamos os 45 nulos identificados no diagnóstico.

In [4]:
# Verifica valores únicos e range
print(f"Tipo: {df['YearsCode'].dtype}")
print(f"Nulos: {df['YearsCode'].isna().sum()}")
print(f"\nEstatísticas:")
print(df['YearsCode'].describe())
print(f"\nMenores valores: {sorted(df['YearsCode'].dropna().unique())[:5]}")
print(f"Maiores valores: {sorted(df['YearsCode'].dropna().unique())[-5:]}")

Tipo: float64
Nulos: 45

Estatísticas:
count    18146.000000
mean        18.197344
std         10.568806
min          1.000000
25%         10.000000
50%         15.000000
75%         25.000000
max        100.000000
Name: YearsCode, dtype: float64

Menores valores: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
Maiores valores: [np.float64(58.0), np.float64(59.0), np.float64(60.0), np.float64(61.0), np.float64(100.0)]


In [5]:
mediana_years = df['YearsCode'].median()
n_acima_50 = (df['YearsCode'] > 50).sum()

df['YearsCode'] = df['YearsCode'].fillna(mediana_years).clip(upper=50)

print(f"Nulos preenchidos com mediana = {mediana_years}")
print(f"Valores acima de 50 anos ajustados: {n_acima_50}")
print(f"\nNovo resumo:")
print(df['YearsCode'].describe())

Nulos preenchidos com mediana = 15.0
Valores acima de 50 anos ajustados: 39

Novo resumo:
count    18191.000000
mean        18.168765
std         10.465099
min          1.000000
25%         10.000000
50%         15.000000
75%         25.000000
max         50.000000
Name: YearsCode, dtype: float64


## 4. Redução de cardinalidade de Country (Top 5 + "Outros")

`Country` tem 153 valores únicos. Reduzimos para os **5 países mais frequentes** + um bucket "Outros". O top 5 é escolhido por **volume de respostas** (não por mediana salarial), porque queremos amostra grande o suficiente para estimação confiável dentro de cada nível.

**Por que esse passo vem antes do tratamento de outliers:** o passo seguinte (§5) remove outliers DENTRO de cada país do Top 5. Para isso, os países precisam estar agrupados primeiro.

In [ ]:
# Identifica top 5 países por frequência
top5_paises = df['Country'].value_counts().head(5).index.tolist()

print("Top 5 países (por frequência na amostra):")
for i, pais in enumerate(top5_paises, 1):
    n = (df['Country'] == pais).sum()
    print(f"  {i:2}. {pais}: {n:,} ({n/len(df)*100:.1f}%)")

In [ ]:
# Cria coluna agrupada (Top 5 + "Outros") e abrevia nomes longos
df['Country_agrupado'] = df['Country'].where(df['Country'].isin(top5_paises), 'Outros')
df['Country_agrupado'] = df['Country_agrupado'].replace({
    'United States of America': 'USA',
    'United Kingdom of Great Britain and Northern Ireland': 'UK',
})

print("Distribuição após agrupamento:")
print(df['Country_agrupado'].value_counts())

## 5. Remoção estratificada de outliers de salário (por país)

**Decisão metodológica:** em vez de aplicar um único corte global (USD 1.000 ≤ salário ≤ USD 500.000), removemos outliers **dentro de cada país do Top 5** usando o critério IQR clássico:

$$
\text{limite\_inf} = Q_1 - 1{,}5 \cdot \text{IQR}, \quad \text{limite\_sup} = Q_3 + 1{,}5 \cdot \text{IQR}
$$

**Por quê estratificado:** a distribuição salarial é radicalmente diferente entre países. Um salário de USD 80.000/ano é mediano nos EUA, mas seria um outlier extremo na Índia (onde a mediana é da ordem de USD 17k); inversamente, um salário de USD 12.000 é normal na Índia, mas seria considerado lixo de dados nos EUA. Um corte global ignora essa heterogeneidade e descarta sinais legítimos de baixa renda em mercados em desenvolvimento, ao mesmo tempo que mantém outliers locais em mercados ricos.

Para o bucket **"Outros"** (países fora do Top 5, com pouca representação individual), aplicamos um filtro global suave (USD 1k–500k) só para tirar lixo extremo — não há amostra suficiente em cada país individual para usar IQR de forma confiável.

A célula seguinte aplica o filtro e gera uma tabela de relatório com Q1, Q3, IQR, limites e número de outliers removidos por país.

In [ ]:
# Constantes para o filtro do bucket "Outros"
LIMITE_GLOBAL_INF = 1_000
LIMITE_GLOBAL_SUP = 500_000

# Vamos coletar as máscaras "manter" para cada grupo e depois concatenar
mascaras_keep = []
relatorio = []

for pais, grupo in df.groupby('Country_agrupado'):
    sal = grupo['ConvertedCompYearly']

    if pais == 'Outros':
        lo, hi = LIMITE_GLOBAL_INF, LIMITE_GLOBAL_SUP
        criterio = 'Global ($1k–$500k)'
    else:
        q1 = sal.quantile(0.25)
        q3 = sal.quantile(0.75)
        iqr = q3 - q1
        lo = max(q1 - 1.5 * iqr, 0)  # impede limite negativo
        hi = q3 + 1.5 * iqr
        criterio = 'IQR (Q1−1.5·IQR, Q3+1.5·IQR)'

    mask_grupo = (sal >= lo) & (sal <= hi)
    mascaras_keep.append(mask_grupo)

    relatorio.append({
        'pais': pais,
        'criterio': criterio,
        'n_antes': len(grupo),
        'n_depois': int(mask_grupo.sum()),
        'removidos': int(len(grupo) - mask_grupo.sum()),
        'pct_removidos': round((len(grupo) - mask_grupo.sum()) / len(grupo) * 100, 2),
        'q1': round(sal.quantile(0.25), 0),
        'q3': round(sal.quantile(0.75), 0),
        'limite_inf': round(lo, 0),
        'limite_sup': round(hi, 0),
        'mediana_pos': round(sal[mask_grupo].median(), 0),
    })

mask_total = pd.concat(mascaras_keep).sort_index()
n_antes = len(df)
df = df[mask_total].copy()
n_depois = len(df)

relatorio_df = (
    pd.DataFrame(relatorio)
      .sort_values('n_antes', ascending=False)
      .reset_index(drop=True)
)

print(f"Antes do filtro:  {n_antes:,} linhas")
print(f"Depois do filtro: {n_depois:,} linhas")
print(f"Total removido:   {n_antes - n_depois:,} ({(n_antes - n_depois)/n_antes*100:.2f}%)")
print()
print("Relatório por país:")
print(relatorio_df.to_string(index=False))

In [9]:
# Identifica top 5 países por frequência
top5_paises = df['Country'].value_counts().head(5).index.tolist()

print("Top 5 países:")
for i, pais in enumerate(top5_paises, 1):
    n = (df['Country'] == pais).sum()
    print(f"  {i:2}. {pais}: {n:,} ({n/len(df)*100:.1f}%)")

# Cria coluna agrupada
df['Country_agrupado'] = df['Country'].where(df['Country'].isin(top5_paises), 'Outros')
# Abrevia nomes longos para gráficos
df['Country_agrupado'] = df['Country_agrupado'].replace({
    'United States of America': 'USA',
    'United Kingdom of Great Britain and Northern Ireland': 'UK',
})
print(f"\nDistribuição após agrupamento:")
print(df['Country_agrupado'].value_counts())

Top 5 países:
   1. United States of America: 3,919 (22.1%)
   2. Germany: 1,583 (8.9%)
   3. United Kingdom of Great Britain and Northern Ireland: 1,153 (6.5%)
   4. France: 775 (4.4%)
   5. India: 770 (4.3%)

Distribuição após agrupamento:
Country_agrupado
Outros     9512
USA        3919
Germany    1583
UK         1153
France      775
India       770
Name: count, dtype: int64


## 6. Agrupamento de DevType em macro-categorias e refinamento da amostra

A coluna original tem 32 valores diferentes. Agrupamos em **5 macro-categorias** relevantes para a pergunta de pesquisa (backend, frontend, fullstack, mobile, data_ml) e **removemos** os respondentes cuja função principal não é desenvolvimento em sentido estrito (engineering managers, founders, senior executives, project managers, academic researchers, students).

**Por que remover em vez de manter como categoria "outros":** a pergunta central do trabalho é sobre desenvolvedores. Cargos de gestão e executivos têm salários determinados por fatores que o modelo não mede equity, prestígio do cargo, tamanho da empresa fundada. Mantê-los na amostra contamina a estimativa dos coeficientes, que é o foco da análise. Excluir é o filtro coerente com a pergunta análogo aos filtros já aplicados no notebook 01 (só devs profissionais, só empregados, só quem respondeu salário).

In [10]:
# Primeiro, ver todos os valores para mapear corretamente
print(df['DevType'].value_counts())

DevType
Developer, full-stack                            6467
Developer, back-end                              3413
Architect, software or solutions                 1235
Developer, desktop or enterprise applications     980
Developer, front-end                              917
Developer, embedded applications or devices       657
Developer, mobile                                 626
Engineering manager                               457
DevOps engineer or professional                   449
Data engineer                                     385
Other (please specify):                           282
AI/ML engineer                                    268
Data scientist                                    202
Cloud infrastructure engineer                     176
Senior executive (C-suite, VP, etc.)              174
Developer, game or graphics                       165
Academic researcher                               132
Founder, technology or otherwise                  119
Developer, QA or tes

In [11]:
# Mapeamento para macro-categorias
mapa_devtype = {
    # Backend
    'Developer, back-end': 'backend',
    'Developer, embedded applications or devices': 'backend',
    'Developer, desktop or enterprise applications': 'backend',

    # Frontend
    'Developer, front-end': 'frontend',

    # Fullstack
    'Developer, full-stack': 'fullstack',

    # Mobile
    'Developer, mobile': 'mobile',

    # Data/ML
    'Data engineer': 'data_ml',
    'Data scientist': 'data_ml',
    'AI/ML engineer': 'data_ml',
    'Data or business analyst': 'data_ml',
    'Applied scientist': 'data_ml',
    'Developer, AI apps or physical AI': 'data_ml',
    'Database administrator or engineer': 'data_ml',
}

# Tudo que não está no mapa vira "outros"
df['DevType_agrupado'] = df['DevType'].map(mapa_devtype).fillna('outros')

print(df['DevType_agrupado'].value_counts())
print(f"\nTotal: {len(df):,}")

DevType_agrupado
fullstack    6467
backend      5050
outros       3541
data_ml      1111
frontend      917
mobile        626
Name: count, dtype: int64

Total: 17,712


In [12]:
# Aplica o filtro: remove respondentes em "outros".
# A categoria agrupa cargos não-desenvolvedores cujos salários respondem
# a fatores fora do escopo do modelo. Mantê-los contaminaria os
# coeficientes das linguagens (foco principal da análise).
n_antes = len(df)
df = df[df['DevType_agrupado'] != 'outros'].copy()
n_depois = len(df)

print(f"Antes do filtro: {n_antes:,} linhas")
print(f"Depois do filtro: {n_depois:,} linhas")
print(f"Removidas: {n_antes - n_depois:,} ({(n_antes - n_depois)/n_antes*100:.2f}%)")
print(f"\nDistribuição final de DevType_agrupado:")
print(df['DevType_agrupado'].value_counts())

Antes do filtro: 17,712 linhas
Depois do filtro: 14,171 linhas
Removidas: 3,541 (19.99%)

Distribuição final de DevType_agrupado:
DevType_agrupado
fullstack    6467
backend      5050
data_ml      1111
frontend      917
mobile        626
Name: count, dtype: int64


## 7. Padronização de EdLevel (top 5 + "Outros")

Encurta rótulos longos para facilitar gráficos e tabelas, e **reduz para os 5 níveis mais frequentes** + "Outros", em coerência com a estratégia de top 5.

In [13]:
mapa_edlevel = {
    'Primary/elementary school': 'Fundamental',
    'Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)': 'Médio',
    'Some college/university study without earning a degree': 'Superior incompleto',
    'Associate degree (A.A., A.S., etc.)': 'Tecnólogo',
    'Bachelor’s degree (B.A., B.S., B.Eng., etc.)': 'Bacharelado',
    'Master’s degree (M.A., M.S., M.Eng., MBA, etc.)': 'Mestrado',
    'Professional degree (JD, MD, Ph.D, Ed.D, etc.)': 'Doutorado',
    'Something else': 'Outros',
}

df['EdLevel_agrupado'] = df['EdLevel'].map(mapa_edlevel).fillna('Outros')

# Reduz para top 5 níveis + "Outros" (em coerência com a estratégia top 5)
top5_edlevel = df['EdLevel_agrupado'].value_counts().head(5).index.tolist()
df['EdLevel_agrupado'] = df['EdLevel_agrupado'].where(df['EdLevel_agrupado'].isin(top5_edlevel), 'Outros')

print("Top 5 níveis de educação + Outros:")
print(df['EdLevel_agrupado'].value_counts())

Top 5 níveis de educação + Outros:
EdLevel_agrupado
Bacharelado            6608
Mestrado               4106
Superior incompleto    1635
Outros                  658
Médio                   622
Doutorado               542
Name: count, dtype: int64


## 8. Dummies das top 5 linguagens

Cria colunas binárias (1/0) para cada uma das **5 linguagens mais usadas** na amostra, baseadas em `LanguageHaveWorkedWith` (multirresposta separada por `;`). Em coerência com a estratégia top 5 aplicada aos demais blocos.

In [14]:
# Gera matriz de dummies (uma coluna por linguagem)
dummies_linguagens = df['LanguageHaveWorkedWith'].str.get_dummies(sep=';')

# Seleciona as top 5 por frequência
top5_linguagens = dummies_linguagens.sum().sort_values(ascending=False).head(5).index.tolist()

print("Top 5 linguagens:")
for i, lang in enumerate(top5_linguagens, 1):
    n = dummies_linguagens[lang].sum()
    print(f"  {i:2}. {lang}: {n:,} ({n/len(df)*100:.1f}%)")

# Renomeia colunas com caracteres problemáticos para o statsmodels
mapa_nomes = {
    'C#': 'CSharp',
    'C++': 'CPlusPlus',
    'HTML/CSS': 'HTML_CSS',
    'Bash/Shell (all shells)': 'Bash_Shell',
}

# Seleciona apenas top 5 e renomeia
dummies_top5 = dummies_linguagens[top5_linguagens].rename(columns=mapa_nomes)

# Prefixo "lang_" para identificar facilmente no modelo
dummies_top5 = dummies_top5.add_prefix('lang_')

# Concatena ao df principal
df = pd.concat([df, dummies_top5], axis=1)

print(f"\nColunas de linguagem criadas: {len(dummies_top5.columns)}")
print(f"Nomes: {dummies_top5.columns.tolist()}")
print(f"\nShape final do df: {df.shape}")

Top 5 linguagens:
   1. JavaScript: 9,912 (69.9%)
   2. HTML/CSS: 9,139 (64.5%)
   3. SQL: 8,939 (63.1%)
   4. Python: 7,381 (52.1%)
   5. TypeScript: 7,246 (51.1%)

Colunas de linguagem criadas: 5
Nomes: ['lang_JavaScript', 'lang_HTML_CSS', 'lang_SQL', 'lang_Python', 'lang_TypeScript']

Shape final do df: (14171, 17)


## 9. Criação de `nivel_experiencia`

Categoriza os anos de experiência em três faixas: Junior (≤4), Pleno (5–7), Sênior (8+). Usado como bloco de comparação no notebook 04 (importância) e como controle alternativo no modelo focado.

In [15]:
def faixa_experiencia(anos: float) -> str:
    if anos <= 4:
        return 'Junior'
    if anos <= 7:
        return 'Pleno'
    return 'Senior'

df['nivel_experiencia'] = df['YearsCode'].apply(faixa_experiencia)

print("Distribuição de nivel_experiencia:")
print(df['nivel_experiencia'].value_counts())

Distribuição de nivel_experiencia:
nivel_experiencia
Senior    12081
Pleno      1530
Junior      560
Name: count, dtype: int64


## 10. Salvamento do dataset limpo

Salva `df_limpo.csv` em `data/processed/` para uso nos próximos notebooks.

In [16]:
caminho_saida = PASTA_DADOS / "df_limpo.csv"
df.to_csv(caminho_saida, index=False)

print(f"Dataset salvo em: {caminho_saida}")
print(f"Linhas: {len(df):,}")
print(f"Colunas: {df.shape[1]}")
print(f"\nColunas finais:")
print(df.columns.tolist())

Dataset salvo em: ..\data\processed\df_limpo.csv
Linhas: 14,171
Colunas: 18

Colunas finais:
['ResponseId', 'MainBranch', 'EdLevel', 'Employment', 'YearsCode', 'DevType', 'Country', 'LanguageHaveWorkedWith', 'ConvertedCompYearly', 'Country_agrupado', 'DevType_agrupado', 'EdLevel_agrupado', 'lang_JavaScript', 'lang_HTML_CSS', 'lang_SQL', 'lang_Python', 'lang_TypeScript', 'nivel_experiencia']


## Conclusões

A etapa de limpeza resultou em um dataset focado em desenvolvedores em sentido estrito, pronto para a análise exploratória e modelagem com a estratégia "dividir e conquistar".

**Tratamentos aplicados:**
- Imputação de 45 valores ausentes de tempo de experiência pela mediana (15 anos) e limitação superior em 50 anos (39 observações ajustadas).
- Remoção de outliers salariais fora do intervalo USD 1.000 a USD 500.000 (479 observações, 2,63% da amostra).
- Salário mantido em USD inteiro (`ConvertedCompYearly`); não usamos escala logarítmica.
- Redução de cardinalidade: 153 países → **top 5 + "Outros"**.
- Agrupamento de 32 categorias de atuação em **5 macro-categorias** relevantes para a pergunta (backend, frontend, fullstack, mobile, data_ml) e remoção dos respondentes em cargos não-desenvolvedores.
- Padronização de escolaridade reduzida para **top 5 + "Outros"**.
- Codificação binária das **5 linguagens mais frequentes** na amostra.
- Criação de `nivel_experiencia` (Junior/Pleno/Senior) para uso opcional como bloco/controle.

O conjunto resultante é a base de todas as análises descritivas e inferenciais subsequentes.